# 🏠 Smart Energy Monitor - Week 7 & 8
## Milestone 4: Flask API + Interactive Web Dashboard

### What We Build:
- ✅ Flask REST API connected to trained LSTM models
- ✅ Interactive HTML/CSS/JS Dashboard
- ✅ Device-wise energy insights & prediction graphs
- ✅ Smart Suggestions feature
- ✅ System architecture documentation
- ✅ Final project report ready

### Prerequisites:
- Week 5-6 completed → `lstm_models/` folder exists with `.h5` files
- Week 5-6 completed → `lstm_models/scalers/` folder with `.pkl` scalers
- `processed_hourly_energy.csv` in current directory

---

## STEP 1: Install Dependencies

In [ ]:
# Install required packages
!pip install flask flask-cors pyngrok --quiet

import os, json, pickle, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print('✓ All packages ready!')
print(f'TensorFlow: {tf.__version__}')

## STEP 2: Load LSTM Models & Data

In [ ]:
# ============================================================
# Load all LSTM models saved in Week 5
# ============================================================

TIME_STEPS = 24
TRAIN_SPLIT = 0.8

# Load CSV data
df = pd.read_csv('processed_hourly_energy.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
appliances = df['Appliance Type'].unique().tolist()

print('='*60)
print('LOADING LSTM MODELS FROM WEEK 5')
print('='*60)

lstm_models = {}
scalers = {}

for appliance in appliances:
    safe_name = appliance.replace(' ', '_').replace('/', '_')
    model_path = f'lstm_models/{safe_name}_model.h5'
    scaler_path = f'lstm_models/{safe_name}_scaler.pkl'

    try:
        lstm_models[appliance] = load_model(model_path, compile=False)
        lstm_models[appliance].compile(optimizer='adam', loss='mse')

        with open(scaler_path, 'rb') as f:
            scalers[appliance] = pickle.load(f)

        print(f'  ✓ {appliance} model loaded')

    except Exception as e:
        print(f'  ✗ {appliance}: {e}')
        # Create fallback scaler if needed
        app_data = df[df['Appliance Type'] == appliance]['Energy Consumption (kWh)'].values.reshape(-1,1)
        scaler = MinMaxScaler()
        scaler.fit(app_data)
        scalers[appliance] = scaler

print(f'\n✓ Loaded {len(lstm_models)} models successfully')
print(f'Appliances: {appliances}')

## STEP 3: Build the Flask API (app.py)

In [ ]:
# ============================================================
# Write the complete Flask app.py to disk
# ============================================================

flask_app_code = '''
import os, json, pickle, warnings
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from flask import Flask, jsonify, request, send_from_directory
from flask_cors import CORS
warnings.filterwarnings("ignore")

import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ─── App Setup ───────────────────────────────────────────────
app = Flask(__name__, static_folder="static", template_folder="templates")
CORS(app)

TIME_STEPS = 24
TRAIN_SPLIT = 0.8

# ─── Load Data & Models ──────────────────────────────────────
df = pd.read_csv("processed_hourly_energy.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"])
APPLIANCES = df["Appliance Type"].unique().tolist()

lstm_models = {}
scalers = {}

for appliance in APPLIANCES:
    safe = appliance.replace(" ", "_").replace("/", "_")
    try:
        m = load_model(f"lstm_models/{safe}_model.h5", compile=False)
        m.compile(optimizer="adam", loss="mse")
        lstm_models[appliance] = m
        with open(f"lstm_models/{safe}_scaler.pkl", "rb") as f:
            scalers[appliance] = pickle.load(f)
    except Exception as e:
        print(f"Warning: Could not load {appliance}: {e}")
        app_data = df[df["Appliance Type"] == appliance]["Energy Consumption (kWh)"].values.reshape(-1,1)
        sc = MinMaxScaler()
        sc.fit(app_data)
        scalers[appliance] = sc

print(f"✓ Loaded {len(lstm_models)} LSTM models")

# ─── Helper Functions ────────────────────────────────────────
def get_appliance_data(appliance):
    data = df[df["Appliance Type"] == appliance].copy()
    return data.sort_values("timestamp").reset_index(drop=True)

def predict_next_hour(appliance, history_24h):
    if appliance not in lstm_models:
        return None
    scaler = scalers[appliance]
    arr = np.array(history_24h).reshape(-1, 1)
    scaled = scaler.transform(arr)
    X = scaled.reshape(1, TIME_STEPS, 1)
    pred_scaled = lstm_models[appliance].predict(X, verbose=0)
    return float(scaler.inverse_transform(pred_scaled)[0][0])

def get_smart_suggestions(stats):
    suggestions = []
    for name, info in stats.items():
        avg = info["avg_kwh"]
        if "Air Conditioning" in name or "Heater" in name:
            if avg > 5:
                suggestions.append({
                    "appliance": name,
                    "type": "high_usage",
                    "icon": "🌡️",
                    "message": f"{name} averages {avg:.1f} kWh/hr. Set temperature to 24-26°C to save up to 25% energy.",
                    "saving": f"~{avg*0.25:.1f} kWh/hr"
                })
        if "Fridge" in name or "Refrigerator" in name:
            suggestions.append({
                "appliance": name,
                "type": "tip",
                "icon": "🧊",
                "message": "Keep fridge temperature at 3-5°C. Avoid placing hot food directly — saves ~15% energy.",
                "saving": f"~{avg*0.15:.1f} kWh/hr"
            })
        if "Washer" in name or "Washing" in name:
            suggestions.append({
                "appliance": name,
                "type": "schedule",
                "icon": "🫧",
                "message": "Run washing machine during off-peak hours (10 PM - 6 AM) to reduce electricity cost.",
                "saving": "Cost saving"
            })
        if "Lighting" in name or "Light" in name:
            suggestions.append({
                "appliance": name,
                "type": "upgrade",
                "icon": "💡",
                "message": "Switch to LED lighting. Saves up to 75% energy compared to incandescent bulbs.",
                "saving": f"~{avg*0.75:.1f} kWh/hr"
            })
    if not suggestions:
        suggestions.append({
            "appliance": "General",
            "type": "tip",
            "icon": "⚡",
            "message": "Your energy usage looks efficient! Keep appliances maintained for optimal performance.",
            "saving": "Ongoing"
        })
    return suggestions

# ─── API Routes ──────────────────────────────────────────────

@app.route("/")
def index():
    return send_from_directory("templates", "index.html")

@app.route("/api/appliances", methods=["GET"])
def get_appliances():
    return jsonify({"appliances": APPLIANCES, "count": len(APPLIANCES)})

@app.route("/api/stats", methods=["GET"])
def get_stats():
    stats = {}
    total_kwh = 0
    for app_name in APPLIANCES:
        data = get_appliance_data(app_name)
        vals = data["Energy Consumption (kWh)"].values
        avg = float(np.mean(vals))
        total_kwh += avg
        stats[app_name] = {
            "avg_kwh": round(avg, 3),
            "max_kwh": round(float(np.max(vals)), 3),
            "min_kwh": round(float(np.min(vals)), 3),
            "total_kwh": round(float(np.sum(vals)), 2),
            "pct_of_total": 0  # filled below
        }
    for name in stats:
        stats[name]["pct_of_total"] = round(stats[name]["avg_kwh"] / total_kwh * 100, 1)
    return jsonify({"stats": stats, "total_avg_kwh": round(total_kwh, 3)})

@app.route("/api/history/<appliance>", methods=["GET"])
def get_history(appliance):
    appliance = appliance.replace("_", " ")
    days = int(request.args.get("days", 7))
    data = get_appliance_data(appliance)
    recent = data.tail(days * 24)
    result = {
        "appliance": appliance,
        "timestamps": recent["timestamp"].dt.strftime("%Y-%m-%d %H:%M").tolist(),
        "values": recent["Energy Consumption (kWh)"].round(3).tolist(),
        "days": days
    }
    return jsonify(result)

@app.route("/api/predict", methods=["POST"])
def predict():
    body = request.json
    appliance = body.get("appliance")
    history = body.get("history")  # Optional: last 24 values

    if appliance not in APPLIANCES:
        return jsonify({"error": f"Unknown appliance: {appliance}"}), 400

    # Use last 24h from dataset if no history given
    if not history:
        data = get_appliance_data(appliance)
        history = data["Energy Consumption (kWh)"].tail(TIME_STEPS).tolist()

    if len(history) < TIME_STEPS:
        return jsonify({"error": f"Need {TIME_STEPS} values, got {len(history)}"}), 400

    history = history[-TIME_STEPS:]
    prediction = predict_next_hour(appliance, history)

    if prediction is None:
        return jsonify({"error": "Model not loaded"}), 500

    return jsonify({
        "appliance": appliance,
        "prediction_kwh": round(max(0, prediction), 4),
        "prediction_time": (datetime.now() + timedelta(hours=1)).strftime("%Y-%m-%d %H:%M"),
        "model": "LSTM"
    })

@app.route("/api/predict_all", methods=["GET"])
def predict_all():
    predictions = {}
    for appliance in APPLIANCES:
        data = get_appliance_data(appliance)
        history = data["Energy Consumption (kWh)"].tail(TIME_STEPS).tolist()
        pred = predict_next_hour(appliance, history)
        predictions[appliance] = {
            "prediction_kwh": round(max(0, pred), 4) if pred else 0,
            "last_actual_kwh": round(history[-1], 4)
        }
    return jsonify({"predictions": predictions, "timestamp": datetime.now().isoformat()})

@app.route("/api/suggestions", methods=["GET"])
def get_suggestions():
    stats_resp = get_stats().get_json()
    suggestions = get_smart_suggestions(stats_resp["stats"])
    return jsonify({"suggestions": suggestions})

@app.route("/api/metrics", methods=["GET"])
def get_metrics():
    metrics = {}
    for appliance in APPLIANCES:
        if appliance not in lstm_models:
            continue
        data = get_appliance_data(appliance)
        vals = data["Energy Consumption (kWh)"].values
        scaler = scalers[appliance]
        split_idx = int(len(vals) * TRAIN_SPLIT)
        test_vals = vals[split_idx:]
        if len(test_vals) < TIME_STEPS + 10:
            continue
        # Quick eval on last 100 test points
        n = min(100, len(test_vals) - TIME_STEPS)
        y_true, y_pred = [], []
        for i in range(n):
            window = test_vals[i:i+TIME_STEPS]
            actual = test_vals[i+TIME_STEPS]
            pred = predict_next_hour(appliance, window.tolist())
            if pred is not None:
                y_true.append(actual)
                y_pred.append(pred)
        if len(y_true) > 5:
            metrics[appliance] = {
                "mae": round(float(mean_absolute_error(y_true, y_pred)), 4),
                "rmse": round(float(np.sqrt(mean_squared_error(y_true, y_pred))), 4),
                "r2": round(float(r2_score(y_true, y_pred)), 4)
            }
    return jsonify({"metrics": metrics})

@app.route("/api/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "models_loaded": len(lstm_models), "appliances": APPLIANCES})

if __name__ == "__main__":
    print("Starting Smart Energy Monitor API...")
    app.run(host="0.0.0.0", port=5000, debug=False)
'''

os.makedirs('smart_energy_app', exist_ok=True)
os.makedirs('smart_energy_app/templates', exist_ok=True)
os.makedirs('smart_energy_app/static', exist_ok=True)

with open('smart_energy_app/app.py', 'w') as f:
    f.write(flask_app_code)

print('✓ app.py written to smart_energy_app/app.py')

## STEP 4: Build the HTML/CSS/JS Dashboard

In [ ]:
# ============================================================
# Write the complete interactive dashboard (index.html)
# ============================================================

html_code = '''<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Smart Energy Monitor</title>
  <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
  <style>
    :root {
      --bg: #0f172a;
      --card: #1e293b;
      --border: #334155;
      --primary: #38bdf8;
      --accent: #818cf8;
      --green: #34d399;
      --yellow: #fbbf24;
      --red: #f87171;
      --text: #e2e8f0;
      --muted: #94a3b8;
    }
    * { margin:0; padding:0; box-sizing:border-box; }
    body { background:var(--bg); color:var(--text); font-family:"Segoe UI",sans-serif; min-height:100vh; }

    /* HEADER */
    header {
      background: linear-gradient(135deg, #1e293b 0%, #0f172a 100%);
      border-bottom: 1px solid var(--border);
      padding: 16px 32px;
      display: flex; align-items:center; justify-content:space-between;
    }
    .logo { display:flex; align-items:center; gap:12px; }
    .logo-icon { font-size:32px; }
    .logo h1 { font-size:22px; font-weight:700; color:var(--primary); }
    .logo span { font-size:13px; color:var(--muted); display:block; }
    .header-status { display:flex; align-items:center; gap:8px; font-size:13px; color:var(--green); }
    .dot { width:8px; height:8px; background:var(--green); border-radius:50%; animation:pulse 2s infinite; }
    @keyframes pulse { 0%,100%{opacity:1} 50%{opacity:.3} }

    /* NAV TABS */
    .tabs {
      display:flex; gap:4px; padding:16px 32px 0;
      border-bottom: 1px solid var(--border);
    }
    .tab {
      padding:10px 20px; border-radius:8px 8px 0 0; cursor:pointer;
      font-size:14px; font-weight:500; color:var(--muted);
      border:1px solid transparent; border-bottom:none;
      transition:all .2s;
    }
    .tab.active { background:var(--card); color:var(--primary); border-color:var(--border); }
    .tab:hover:not(.active) { color:var(--text); background:rgba(255,255,255,.05); }

    /* MAIN */
    main { padding:24px 32px; }
    .panel { display:none; }
    .panel.active { display:block; }

    /* STATS GRID */
    .stats-grid { display:grid; grid-template-columns:repeat(auto-fill,minmax(200px,1fr)); gap:16px; margin-bottom:24px; }
    .stat-card {
      background:var(--card); border:1px solid var(--border); border-radius:12px;
      padding:20px; transition:transform .2s;
    }
    .stat-card:hover { transform:translateY(-2px); }
    .stat-card .icon { font-size:28px; margin-bottom:8px; }
    .stat-card .label { font-size:12px; color:var(--muted); text-transform:uppercase; letter-spacing:.5px; }
    .stat-card .value { font-size:26px; font-weight:700; color:var(--primary); margin:4px 0; }
    .stat-card .sub { font-size:12px; color:var(--muted); }
    .stat-card .bar-bg { background:var(--border); border-radius:4px; height:4px; margin-top:10px; }
    .stat-card .bar { background:var(--primary); height:4px; border-radius:4px; transition:width .8s; }

    /* CHARTS */
    .chart-section { background:var(--card); border:1px solid var(--border); border-radius:12px; padding:20px; margin-bottom:20px; }
    .chart-header { display:flex; align-items:center; justify-content:space-between; margin-bottom:16px; }
    .chart-title { font-size:16px; font-weight:600; }
    .chart-controls { display:flex; gap:8px; }
    select, button {
      background:var(--border); color:var(--text); border:1px solid var(--border);
      border-radius:6px; padding:6px 12px; font-size:13px; cursor:pointer;
    }
    button:hover { background:var(--primary); color:#000; }
    select:focus { outline:2px solid var(--primary); }
    canvas { max-height:300px; }

    /* TWO COLUMN LAYOUT */
    .two-col { display:grid; grid-template-columns:1fr 1fr; gap:20px; margin-bottom:20px; }
    @media(max-width:800px){ .two-col{grid-template-columns:1fr;} }

    /* PREDICTIONS TABLE */
    .pred-table { width:100%; border-collapse:collapse; font-size:14px; }
    .pred-table th { background:var(--border); padding:10px 14px; text-align:left; font-weight:600; color:var(--muted); font-size:12px; text-transform:uppercase; }
    .pred-table td { padding:10px 14px; border-bottom:1px solid var(--border); }
    .pred-table tr:hover td { background:rgba(255,255,255,.03); }
    .badge { padding:3px 8px; border-radius:12px; font-size:11px; font-weight:600; }
    .badge-up { background:rgba(52,211,153,.15); color:var(--green); }
    .badge-down { background:rgba(248,113,113,.15); color:var(--red); }
    .badge-neutral { background:rgba(251,191,36,.15); color:var(--yellow); }

    /* SUGGESTIONS */
    .suggestion-list { display:flex; flex-direction:column; gap:12px; }
    .suggestion-card {
      background:var(--card); border:1px solid var(--border); border-radius:12px;
      padding:16px 20px; display:flex; align-items:flex-start; gap:16px;
    }
    .suggestion-icon { font-size:28px; flex-shrink:0; }
    .suggestion-body h3 { font-size:15px; font-weight:600; margin-bottom:4px; }
    .suggestion-body p { font-size:13px; color:var(--muted); line-height:1.5; }
    .suggestion-saving { margin-top:8px; font-size:12px; color:var(--green); font-weight:600; }
    .tag { display:inline-block; padding:2px 8px; border-radius:10px; font-size:11px; margin-right:6px; }
    .tag-high { background:rgba(248,113,113,.15); color:var(--red); }
    .tag-tip { background:rgba(129,140,248,.15); color:var(--accent); }
    .tag-schedule { background:rgba(251,191,36,.15); color:var(--yellow); }
    .tag-upgrade { background:rgba(52,211,153,.15); color:var(--green); }

    /* METRICS */
    .metrics-grid { display:grid; grid-template-columns:repeat(auto-fill,minmax(280px,1fr)); gap:16px; }
    .metric-card { background:var(--card); border:1px solid var(--border); border-radius:12px; padding:20px; }
    .metric-card h3 { font-size:15px; font-weight:600; margin-bottom:16px; color:var(--primary); }
    .metric-row { display:flex; justify-content:space-between; padding:8px 0; border-bottom:1px solid var(--border); }
    .metric-row:last-child { border-bottom:none; }
    .metric-label { font-size:13px; color:var(--muted); }
    .metric-val { font-size:13px; font-weight:600; }
    .good { color:var(--green); } .ok { color:var(--yellow); } .bad { color:var(--red); }

    /* LOADING */
    .loading { text-align:center; padding:40px; color:var(--muted); }
    .spinner { width:32px; height:32px; border:3px solid var(--border); border-top-color:var(--primary); border-radius:50%; animation:spin 1s linear infinite; margin:0 auto 12px; }
    @keyframes spin { to{transform:rotate(360deg)} }

    /* ARCH DIAGRAM */
    .arch-box { background:var(--card); border:1px solid var(--border); border-radius:12px; padding:24px; margin-bottom:20px; }
    .arch-flow { display:flex; align-items:center; justify-content:center; gap:0; flex-wrap:wrap; margin:20px 0; }
    .arch-node {
      background:var(--border); border-radius:10px; padding:14px 20px;
      text-align:center; min-width:130px; font-size:13px;
    }
    .arch-node .node-icon { font-size:24px; display:block; margin-bottom:4px; }
    .arch-node .node-title { font-weight:600; display:block; color:var(--primary); }
    .arch-node .node-sub { color:var(--muted); font-size:11px; }
    .arch-arrow { font-size:20px; color:var(--muted); padding:0 8px; }
    .arch-table { width:100%; border-collapse:collapse; font-size:13px; }
    .arch-table th { background:var(--border); padding:8px 12px; text-align:left; }
    .arch-table td { padding:8px 12px; border-bottom:1px solid var(--border); }
    .arch-table tr:hover td { background:rgba(255,255,255,.02); }
  </style>
</head>
<body>

<header>
  <div class="logo">
    <div class="logo-icon">⚡</div>
    <div>
      <h1>Smart Energy Monitor</h1>
      <span>LSTM-Powered Energy Forecasting Dashboard</span>
    </div>
  </div>
  <div class="header-status">
    <div class="dot"></div>
    <span id="statusText">Connecting...</span>
  </div>
</header>

<div class="tabs">
  <div class="tab active" onclick="showPanel(\'overview\',this)">📊 Overview</div>
  <div class="tab" onclick="showPanel(\'devices\',this)">🔌 Device Insights</div>
  <div class="tab" onclick="showPanel(\'predictions\',this)">🔮 Predictions</div>
  <div class="tab" onclick="showPanel(\'suggestions\',this)">💡 Smart Suggestions</div>
  <div class="tab" onclick="showPanel(\'metrics\',this)">📈 Model Metrics</div>
  <div class="tab" onclick="showPanel(\'architecture\',this)">🏗️ Architecture</div>
</div>

<main>

  <!-- OVERVIEW PANEL -->
  <div id="panel-overview" class="panel active">
    <div id="statsGrid" class="stats-grid"><div class="loading"><div class="spinner"></div>Loading stats...</div></div>
    <div class="two-col">
      <div class="chart-section">
        <div class="chart-header">
          <span class="chart-title">Total Energy Consumption (All Appliances)</span>
        </div>
        <canvas id="overviewChart"></canvas>
      </div>
      <div class="chart-section">
        <div class="chart-header">
          <span class="chart-title">Energy Share by Device</span>
        </div>
        <canvas id="pieChart"></canvas>
      </div>
    </div>
  </div>

  <!-- DEVICE INSIGHTS PANEL -->
  <div id="panel-devices" class="panel">
    <div class="chart-section">
      <div class="chart-header">
        <span class="chart-title">Device Energy History</span>
        <div class="chart-controls">
          <select id="applianceSelect" onchange="loadDeviceChart()"></select>
          <select id="daysSelect" onchange="loadDeviceChart()">
            <option value="3">3 days</option>
            <option value="7" selected>7 days</option>
            <option value="14">14 days</option>
            <option value="30">30 days</option>
          </select>
        </div>
      </div>
      <canvas id="deviceChart"></canvas>
    </div>
    <div id="deviceStats" class="two-col"></div>
  </div>

  <!-- PREDICTIONS PANEL -->
  <div id="panel-predictions" class="panel">
    <div class="chart-section">
      <div class="chart-header">
        <span class="chart-title">LSTM Next-Hour Predictions vs Last Actual</span>
        <button onclick="loadPredictions()">🔄 Refresh</button>
      </div>
      <canvas id="predChart"></canvas>
    </div>
    <div class="chart-section">
      <div class="chart-header"><span class="chart-title">Prediction Summary Table</span></div>
      <table class="pred-table">
        <thead><tr><th>Appliance</th><th>Last Actual (kWh)</th><th>LSTM Prediction (kWh)</th><th>Change</th><th>Prediction Time</th></tr></thead>
        <tbody id="predTableBody"><tr><td colspan="5" class="loading">Loading...</td></tr></tbody>
      </table>
    </div>
  </div>

  <!-- SUGGESTIONS PANEL -->
  <div id="panel-suggestions" class="panel">
    <div style="margin-bottom:16px;">
      <h2 style="font-size:18px; font-weight:600;">💡 Smart Energy Saving Suggestions</h2>
      <p style="color:var(--muted); font-size:14px; margin-top:4px;">Personalized recommendations based on your usage patterns</p>
    </div>
    <div id="suggestionList" class="suggestion-list"><div class="loading"><div class="spinner"></div>Analyzing usage patterns...</div></div>
  </div>

  <!-- METRICS PANEL -->
  <div id="panel-metrics" class="panel">
    <div style="margin-bottom:16px;">
      <h2 style="font-size:18px; font-weight:600;">📈 LSTM Model Performance Metrics</h2>
      <p style="color:var(--muted); font-size:14px; margin-top:4px;">Evaluated on test set (20% holdout)</p>
    </div>
    <div id="metricsGrid" class="metrics-grid"><div class="loading"><div class="spinner"></div>Computing metrics...</div></div>
  </div>

  <!-- ARCHITECTURE PANEL -->
  <div id="panel-architecture" class="panel">
    <div class="arch-box">
      <h2 style="font-size:18px; font-weight:600; margin-bottom:8px;">🏗️ System Architecture</h2>
      <p style="color:var(--muted); font-size:14px;">End-to-end Smart Energy Monitor workflow</p>
      <div class="arch-flow">
        <div class="arch-node"><span class="node-icon">📂</span><span class="node-title">Data Source</span><span class="node-sub">processed_hourly_energy.csv</span></div>
        <div class="arch-arrow">→</div>
        <div class="arch-node"><span class="node-icon">🧹</span><span class="node-title">Preprocessing</span><span class="node-sub">MinMaxScaler + Sequences</span></div>
        <div class="arch-arrow">→</div>
        <div class="arch-node"><span class="node-icon">🧠</span><span class="node-title">LSTM Model</span><span class="node-sub">64→32 units, Dropout</span></div>
        <div class="arch-arrow">→</div>
        <div class="arch-node"><span class="node-icon">🐍</span><span class="node-title">Flask API</span><span class="node-sub">/api/predict, /api/stats</span></div>
        <div class="arch-arrow">→</div>
        <div class="arch-node"><span class="node-icon">🖥️</span><span class="node-title">Dashboard</span><span class="node-sub">HTML + Chart.js + JS</span></div>
      </div>
    </div>
    <div class="arch-box">
      <h3 style="font-size:16px; font-weight:600; margin-bottom:12px;">📋 API Endpoints</h3>
      <table class="arch-table">
        <thead><tr><th>Method</th><th>Endpoint</th><th>Description</th></tr></thead>
        <tbody>
          <tr><td>GET</td><td>/api/health</td><td>Check API status and loaded models</td></tr>
          <tr><td>GET</td><td>/api/appliances</td><td>List all available appliances</td></tr>
          <tr><td>GET</td><td>/api/stats</td><td>Usage stats per appliance (avg, max, total)</td></tr>
          <tr><td>GET</td><td>/api/history/{appliance}?days=7</td><td>Historical data for charts</td></tr>
          <tr><td>POST</td><td>/api/predict</td><td>Predict next hour for one appliance</td></tr>
          <tr><td>GET</td><td>/api/predict_all</td><td>Predictions for all appliances</td></tr>
          <tr><td>GET</td><td>/api/suggestions</td><td>Smart energy-saving suggestions</td></tr>
          <tr><td>GET</td><td>/api/metrics</td><td>MAE, RMSE, R² for all models</td></tr>
        </tbody>
      </table>
    </div>
    <div class="arch-box">
      <h3 style="font-size:16px; font-weight:600; margin-bottom:12px;">🗂️ Project Structure</h3>
      <pre style="color:var(--muted); font-size:13px; line-height:1.8;">
smart_energy_monitor/
├── app.py                     ← Flask API (8 endpoints)
├── processed_hourly_energy.csv ← Dataset
├── lstm_models/               ← Trained LSTM models (from Week 5)
│   ├── Air_Conditioning_model.h5
│   ├── Air_Conditioning_scaler.pkl
│   └── ... (one per appliance)
├── templates/
│   └── index.html             ← Interactive Dashboard (this page)
└── static/                    ← CSS/JS assets
      </pre>
    </div>
    <div class="arch-box">
      <h3 style="font-size:16px; font-weight:600; margin-bottom:12px;">📊 ML Pipeline Summary</h3>
      <table class="arch-table">
        <thead><tr><th>Stage</th><th>Details</th></tr></thead>
        <tbody>
          <tr><td>Data Source</td><td>IoT sensor data (household appliances, hourly readings)</td></tr>
          <tr><td>Preprocessing</td><td>Missing value handling, MinMaxScaler normalization</td></tr>
          <tr><td>Sequence Length</td><td>24 hours (used as input window)</td></tr>
          <tr><td>Train/Test Split</td><td>80% / 20% chronological</td></tr>
          <tr><td>Model Architecture</td><td>LSTM(64) → Dropout(0.2) → LSTM(32) → Dropout(0.2) → Dense(1)</td></tr>
          <tr><td>Optimizer</td><td>Adam (lr=0.001)</td></tr>
          <tr><td>Loss Function</td><td>Mean Squared Error (MSE)</td></tr>
          <tr><td>Callbacks</td><td>EarlyStopping, ReduceLROnPlateau, ModelCheckpoint</td></tr>
          <tr><td>Baseline Comparison</td><td>LSTM vs Linear Regression (Week 3-4)</td></tr>
          <tr><td>Deployment</td><td>Flask REST API + HTML/JS Dashboard</td></tr>
        </tbody>
      </table>
    </div>
  </div>

</main>

<script>
// ── CONFIG ─────────────────────────────────────────────────
const API = \'\'; // Empty = same origin. For ngrok: const API = \'https://YOUR-NGROK-URL\'

let overviewChart, pieChart, deviceChart, predChart;
let applianceList = [];

// ── UTILITIES ──────────────────────────────────────────────
async function apiFetch(path) {
  const res = await fetch(API + path);
  return res.json();
}

function showPanel(id, el) {
  document.querySelectorAll(\'.panel\').forEach(p => p.classList.remove(\'active\'));
  document.querySelectorAll(\'.tab\').forEach(t => t.classList.remove(\'active\'));
  document.getElementById(\'panel-\' + id).classList.add(\'active\');
  el.classList.add(\'active\');
  if (id === \'devices\') loadDeviceChart();
  if (id === \'predictions\') loadPredictions();
  if (id === \'suggestions\') loadSuggestions();
  if (id === \'metrics\') loadMetrics();
}

function getColor(i, a=1) {
  const colors = [\'56,189,248\',\'129,140,248\',\'52,211,153\',\'251,191,36\',\'248,113,113\',\'167,139,250\',\'34,211,238\',\'251,146,60\',\'163,230,53\',\'244,114,182\'];
  return `rgba(${colors[i % colors.length]},${a})`;
}

function destroyChart(ref) { if (ref) { ref.destroy(); } return null; }

// ── INIT ───────────────────────────────────────────────────
async function init() {
  try {
    const h = await apiFetch(\'/api/health\');
    document.getElementById(\'statusText\').textContent = `${h.models_loaded} models loaded`;
    applianceList = h.appliances;

    // Fill appliance select
    const sel = document.getElementById(\'applianceSelect\');
    applianceList.forEach(a => {
      const o = document.createElement(\'option\'); o.value=a; o.textContent=a; sel.appendChild(o);
    });

    loadOverview();
  } catch(e) {
    document.getElementById(\'statusText\').textContent = \'API offline - check Flask server\';
    document.getElementById(\'statsGrid\').innerHTML = \'<p style="color:var(--red);padding:20px">⚠️ Cannot connect to Flask API. Make sure the server is running (see Step 5 of notebook).</p>\';
  }
}

// ── OVERVIEW ───────────────────────────────────────────────
async function loadOverview() {
  const data = await apiFetch(\'/api/stats\');
  const stats = data.stats;
  const appls = Object.keys(stats);

  // Stat cards
  const icons = {\'Air Conditioning\':\'❄️\', \'Heater\':\'🔥\', \'Fridge\':\'🧊\', \'Refrigerator\':\'🧊\', \'Washer\':\'🫧\', \'Washing Machine\':\'🫧\', \'Lighting\':\'💡\', \'TV\':\'📺\', \'Dishwasher\':\'🍽️\', default:\'🔌\'};
  let html = \'\';
  appls.forEach((a, i) => {
    const s = stats[a];
    const icon = icons[a] || icons.default;
    html += `<div class="stat-card">
      <div class="icon">${icon}</div>
      <div class="label">${a}</div>
      <div class="value">${s.avg_kwh}</div>
      <div class="sub">avg kWh/hr • ${s.pct_of_total}% of total</div>
      <div class="bar-bg"><div class="bar" style="width:${s.pct_of_total}%"></div></div>
    </div>`;
  });
  document.getElementById(\'statsGrid\').innerHTML = html;

  // Overview line chart - last 7 days for first appliance
  const hist = await apiFetch(`/api/history/${encodeURIComponent(appls[0])}?days=7`);
  const ctx = document.getElementById(\'overviewChart\').getContext(\'2d\');
  overviewChart = destroyChart(overviewChart);
  overviewChart = new Chart(ctx, {
    type: \'line\',
    data: {
      labels: hist.timestamps.filter((_,i)=>i%4===0),
      datasets: [{
        label: appls[0],
        data: hist.values.filter((_,i)=>i%4===0),
        borderColor: getColor(0), backgroundColor: getColor(0,.1),
        fill: true, tension: 0.4, pointRadius: 1
      }]
    },
    options: { responsive:true, plugins:{legend:{labels:{color:\'#94a3b8\'}}}, scales:{ x:{ticks:{color:\'#64748b\',maxRotation:45,maxTicksLimit:8}}, y:{ticks:{color:\'#64748b\'}, grid:{color:\'#1e293b\'}} } }
  });

  // Pie chart
  const ctx2 = document.getElementById(\'pieChart\').getContext(\'2d\');
  pieChart = destroyChart(pieChart);
  pieChart = new Chart(ctx2, {
    type: \'doughnut\',
    data: {
      labels: appls,
      datasets: [{ data: appls.map(a=>stats[a].avg_kwh), backgroundColor: appls.map((_,i)=>getColor(i,.8)), borderColor: \'#1e293b\', borderWidth: 2 }]
    },
    options: { responsive:true, plugins:{legend:{position:\'right\',labels:{color:\'#94a3b8\',font:{size:11}}}} }
  });
}

// ── DEVICE CHART ───────────────────────────────────────────
async function loadDeviceChart() {
  const appliance = document.getElementById(\'applianceSelect\').value;
  const days = document.getElementById(\'daysSelect\').value;
  if (!appliance) return;

  const hist = await apiFetch(`/api/history/${encodeURIComponent(appliance)}?days=${days}`);
  const step = Math.max(1, Math.floor(hist.values.length/80));
  const labels = hist.timestamps.filter((_,i)=>i%step===0);
  const values = hist.values.filter((_,i)=>i%step===0);

  const idx = applianceList.indexOf(appliance);
  const ctx = document.getElementById(\'deviceChart\').getContext(\'2d\');
  deviceChart = destroyChart(deviceChart);
  deviceChart = new Chart(ctx, {
    type: \'line\',
    data: {
      labels,
      datasets: [{
        label: appliance + \' (kWh)\',
        data: values,
        borderColor: getColor(idx), backgroundColor: getColor(idx,.1),
        fill: true, tension: 0.4, pointRadius: 1
      }]
    },
    options: { responsive:true, plugins:{legend:{labels:{color:\'#94a3b8\'}}}, scales:{ x:{ticks:{color:\'#64748b\',maxRotation:45,maxTicksLimit:10}}, y:{ticks:{color:\'#64748b\'}, grid:{color:\'#1e293b\'}} } }
  });

  // Device stat cards
  const statsData = await apiFetch(\'/api/stats\');
  const s = statsData.stats[appliance];
  if (!s) return;
  document.getElementById(\'deviceStats\').innerHTML = `
    <div class="chart-section">
      <div class="chart-title" style="margin-bottom:16px">${appliance} - Statistics</div>
      <div class="metric-row"><span class="metric-label">Average Consumption</span><span class="metric-val">${s.avg_kwh} kWh/hr</span></div>
      <div class="metric-row"><span class="metric-label">Peak Consumption</span><span class="metric-val">${s.max_kwh} kWh/hr</span></div>
      <div class="metric-row"><span class="metric-label">Minimum Consumption</span><span class="metric-val">${s.min_kwh} kWh/hr</span></div>
      <div class="metric-row"><span class="metric-label">Total Recorded</span><span class="metric-val">${s.total_kwh} kWh</span></div>
    </div>
    <div class="chart-section">
      <div class="chart-title" style="margin-bottom:16px">Share of Total Usage</div>
      <p style="font-size:40px;font-weight:700;color:var(--primary);text-align:center;margin:20px 0">${s.pct_of_total}%</p>
      <p style="text-align:center;color:var(--muted);font-size:13px">of your total average energy consumption</p>
    </div>`;
}

// ── PREDICTIONS ────────────────────────────────────────────
async function loadPredictions() {
  document.getElementById(\'predTableBody\').innerHTML = \'<tr><td colspan="5" class="loading"><div class="spinner" style="margin:0 auto"></div></td></tr>\';
  const data = await apiFetch(\'/api/predict_all\');
  const preds = data.predictions;
  const appls = Object.keys(preds);
  const predTime = new Date(Date.now() + 3600000).toLocaleTimeString([], {hour:\'2-digit\',minute:\'2-digit\'});

  // Table
  let rows = \'\';
  appls.forEach(a => {
    const p = preds[a];
    const diff = p.prediction_kwh - p.last_actual_kwh;
    const pct = ((diff / (p.last_actual_kwh||1)) * 100).toFixed(1);
    const badge = diff > 0.05 ? `<span class="badge badge-up">▲ +${pct}%</span>` :
                  diff < -0.05 ? `<span class="badge badge-down">▼ ${pct}%</span>` :
                  `<span class="badge badge-neutral">≈ stable</span>`;
    rows += `<tr><td>${a}</td><td>${p.last_actual_kwh} kWh</td><td><strong>${p.prediction_kwh} kWh</strong></td><td>${badge}</td><td>${predTime}</td></tr>`;
  });
  document.getElementById(\'predTableBody\').innerHTML = rows;

  // Comparison chart
  const ctx = document.getElementById(\'predChart\').getContext(\'2d\');
  predChart = destroyChart(predChart);
  predChart = new Chart(ctx, {
    type: \'bar\',
    data: {
      labels: appls,
      datasets: [
        { label: \'Last Actual (kWh)\', data: appls.map(a=>preds[a].last_actual_kwh), backgroundColor: getColor(0,.7), borderColor: getColor(0), borderWidth:1 },
        { label: \'LSTM Prediction (kWh)\', data: appls.map(a=>preds[a].prediction_kwh), backgroundColor: getColor(2,.7), borderColor: getColor(2), borderWidth:1 }
      ]
    },
    options: { responsive:true, plugins:{legend:{labels:{color:\'#94a3b8\'}}}, scales:{ x:{ticks:{color:\'#64748b\'}}, y:{ticks:{color:\'#64748b\'}, grid:{color:\'#1e293b\'}} } }
  });
}

// ── SUGGESTIONS ────────────────────────────────────────────
async function loadSuggestions() {
  const data = await apiFetch(\'/api/suggestions\');
  const tagClass = {high_usage:\'tag-high\',tip:\'tag-tip\',schedule:\'tag-schedule\',upgrade:\'tag-upgrade\'};
  let html = \'\';
  data.suggestions.forEach(s => {
    html += `<div class="suggestion-card">
      <div class="suggestion-icon">${s.icon}</div>
      <div class="suggestion-body">
        <h3>${s.appliance} <span class="tag ${tagClass[s.type]||\'\'} ">${s.type.replace(\'_\',' \')}</span></h3>
        <p>${s.message}</p>
        <div class="suggestion-saving">💚 Potential saving: ${s.saving}</div>
      </div>
    </div>`;
  });
  document.getElementById(\'suggestionList\').innerHTML = html || \'<p style="color:var(--muted)">No suggestions available.</p>\';
}

// ── METRICS ────────────────────────────────────────────────
async function loadMetrics() {
  document.getElementById(\'metricsGrid\').innerHTML = \'<div class="loading"><div class="spinner"></div>Computing (may take 30s)...</div>\';
  const data = await apiFetch(\'/api/metrics\');
  const metrics = data.metrics;
  let html = \'\';
  Object.entries(metrics).forEach(([app, m]) => {
    const r2Class = m.r2 >= 0.8 ? \'good\' : m.r2 >= 0.6 ? \'ok\' : \'bad\';
    html += `<div class="metric-card">
      <h3>${app}</h3>
      <div class="metric-row"><span class="metric-label">MAE</span><span class="metric-val">${m.mae} kWh</span></div>
      <div class="metric-row"><span class="metric-label">RMSE</span><span class="metric-val">${m.rmse} kWh</span></div>
      <div class="metric-row"><span class="metric-label">R² Score</span><span class="metric-val ${r2Class}">${m.r2} ${m.r2>=0.8?\'✓\':\'\'}</span></div>
    </div>`;
  });
  document.getElementById(\'metricsGrid\').innerHTML = html || \'<p style="color:var(--muted);padding:20px">No metrics available. Models may not be loaded.</p>\';
}

// ── START ──────────────────────────────────────────────────
init();
</script>
</body>
</html>'''

with open('smart_energy_app/templates/index.html', 'w') as f:
    f.write(html_code)

print('✓ index.html dashboard written to smart_energy_app/templates/index.html')
print(f'Dashboard size: {len(html_code):,} characters')

## STEP 5: Copy Data & Models into App Folder, Then Run the Server

In [ ]:
import shutil, os

# Copy dataset to app folder
shutil.copy('processed_hourly_energy.csv', 'smart_energy_app/processed_hourly_energy.csv')

# Copy lstm_models to app folder
if os.path.exists('lstm_models'):
    if os.path.exists('smart_energy_app/lstm_models'):
        shutil.rmtree('smart_energy_app/lstm_models')
    shutil.copytree('lstm_models', 'smart_energy_app/lstm_models')
    print('✓ lstm_models copied')
else:
    print('⚠️  lstm_models/ not found - make sure Week 5 ran successfully first!')

print('\nApp folder contents:')
for root, dirs, files in os.walk('smart_energy_app'):
    level = root.replace('smart_energy_app', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        size = os.path.getsize(os.path.join(root, file))
        print(f'{subindent}{file} ({size/1024:.1f} KB)')

## STEP 6: Launch Flask Server with ngrok (Public URL)

In [ ]:
# ============================================================
# Launch Flask + expose via ngrok for public access
# ============================================================
# NOTE: Get your free ngrok token at https://dashboard.ngrok.com/signup
# Then paste it below

NGROK_TOKEN = 'YOUR_NGROK_AUTH_TOKEN_HERE'  # <-- PASTE YOUR TOKEN HERE

import subprocess, threading, time
from pyngrok import ngrok, conf

# Set ngrok auth token
if NGROK_TOKEN != 'YOUR_NGROK_AUTH_TOKEN_HERE':
    conf.get_default().auth_token = NGROK_TOKEN
    ngrok.set_auth_token(NGROK_TOKEN)

# Change to app directory and start Flask in background
os.chdir('smart_energy_app')

def run_flask():
    os.system('python app.py')

# Start Flask in a background thread
flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()

# Wait for Flask to start
time.sleep(4)

# Open ngrok tunnel
if NGROK_TOKEN != 'YOUR_NGROK_AUTH_TOKEN_HERE':
    public_url = ngrok.connect(5000).public_url
    print('='*60)
    print('🚀 SMART ENERGY MONITOR IS LIVE!')
    print('='*60)
    print(f'\n🌐 Dashboard URL: {public_url}')
    print(f'\n📡 API Endpoints:')
    print(f'  {public_url}/api/health')
    print(f'  {public_url}/api/stats')
    print(f'  {public_url}/api/predict_all')
    print(f'  {public_url}/api/suggestions')
    print(f'  {public_url}/api/metrics')
    print('\n✓ Open the Dashboard URL in your browser!')
else:
    print('='*60)
    print('Flask running at http://localhost:5000')
    print('To get public URL: paste your ngrok token above')
    print('Get free token: https://dashboard.ngrok.com/signup')
    print('='*60)

## STEP 7: Test All API Endpoints

In [ ]:
import requests

BASE = 'http://localhost:5000'

print('='*60)
print('TESTING FLASK API ENDPOINTS')
print('='*60)

# 1. Health check
r = requests.get(f'{BASE}/api/health')
print(f'\n1. /api/health: {r.status_code}')
print(f'   {r.json()}')

# 2. Appliances
r = requests.get(f'{BASE}/api/appliances')
print(f'\n2. /api/appliances: {r.status_code}')
d = r.json()
print(f'   Count: {d["count"]}, Appliances: {d["appliances"]}')

# 3. Stats
r = requests.get(f'{BASE}/api/stats')
print(f'\n3. /api/stats: {r.status_code}')
d = r.json()
for app, s in list(d['stats'].items())[:3]:
    print(f'   {app}: avg={s["avg_kwh"]} kWh, {s["pct_of_total"]}% of total')

# 4. History
appliance = list(d['stats'].keys())[0]
r = requests.get(f'{BASE}/api/history/{appliance}?days=3')
print(f'\n4. /api/history/{appliance}?days=3: {r.status_code}')
hd = r.json()
print(f'   Points returned: {len(hd["values"])}')

# 5. Predict
r = requests.post(f'{BASE}/api/predict', json={'appliance': appliance})
print(f'\n5. POST /api/predict: {r.status_code}')
print(f'   {r.json()}')

# 6. Predict all
r = requests.get(f'{BASE}/api/predict_all')
print(f'\n6. /api/predict_all: {r.status_code}')
pd2 = r.json()
for app, p in list(pd2['predictions'].items())[:3]:
    print(f'   {app}: predicted={p["prediction_kwh"]} kWh')

# 7. Suggestions
r = requests.get(f'{BASE}/api/suggestions')
print(f'\n7. /api/suggestions: {r.status_code}')
for s in r.json()['suggestions'][:2]:
    print(f'   {s["icon"]} [{s["appliance"]}]: {s["message"][:60]}...')

print('\n' + '='*60)
print('✓ ALL API ENDPOINTS WORKING!')
print('='*60)

## STEP 8: Final Project Report Summary

In [ ]:
import requests, json
BASE = 'http://localhost:5000'

print('='*70)
print('FINAL PROJECT REPORT - SMART ENERGY MONITOR')
print('='*70)

print('''
PROJECT OVERVIEW
─────────────────
Title     : Smart Energy Monitor using LSTM Deep Learning
Goal      : Predict hourly energy consumption per appliance
Dataset   : Household IoT sensor data (processed_hourly_energy.csv)
Duration  : 8 weeks (Milestones 1-4)
''')

print('''
MILESTONE SUMMARY
──────────────────
Week 1-2  [Milestone 1] : Data collection, EDA, problem framing
Week 3-4  [Milestone 2] : Linear Regression baseline (scikit-learn)
Week 5-6  [Milestone 3] : LSTM Deep Learning model (TensorFlow/Keras)
Week 7-8  [Milestone 4] : Flask API + Interactive Web Dashboard ← YOU ARE HERE
''')

print('''
TECHNICAL STACK
────────────────
Language     : Python 3.x
ML Framework : TensorFlow 2.x / Keras
API          : Flask + Flask-CORS
Frontend     : HTML5, CSS3, Vanilla JavaScript
Charts       : Chart.js (CDN)
Deployment   : Local + ngrok tunnel (Colab-friendly)
''')

# Fetch live metrics from running API
try:
    metrics = requests.get(f'{BASE}/api/metrics').json()['metrics']
    stats_data = requests.get(f'{BASE}/api/stats').json()

    print('MODEL PERFORMANCE (LSTM on Test Set)')
    print('─────────────────────────────────────')
    print(f'{"Appliance":<25} {"MAE":>8} {"RMSE":>8} {"R²":>8}')
    print('-' * 55)
    r2_vals = []
    for app, m in metrics.items():
        r2_vals.append(m['r2'])
        rating = '✓ Excellent' if m['r2']>=0.85 else '✓ Good' if m['r2']>=0.70 else '~ Fair'
        print(f'{app:<25} {m["mae"]:>8.4f} {m["rmse"]:>8.4f} {m["r2"]:>8.4f}  {rating}')

    if r2_vals:
        print(f'\nAverage R²: {sum(r2_vals)/len(r2_vals):.4f}')
        print(f'Best model: {max(metrics, key=lambda x: metrics[x]["r2"])} (R²={max(m["r2"] for m in metrics.values()):.4f})')

    print()
    print('APPLIANCE USAGE INSIGHTS')
    print('─────────────────────────')
    stats = stats_data['stats']
    sorted_apps = sorted(stats.items(), key=lambda x: x[1]['avg_kwh'], reverse=True)
    for app, s in sorted_apps:
        bar = '█' * int(s['pct_of_total'] / 2)
        print(f'{app:<25} {s["avg_kwh"]:>6.3f} kWh/hr  {bar} {s["pct_of_total"]}%')

except Exception as e:
    print(f'Could not fetch live metrics: {e}')
    print('(Run Flask server first in Step 6)')

print('''
DASHBOARD FEATURES
───────────────────
[1] Overview Tab      : Summary stats + line chart + pie chart
[2] Device Insights   : Per-appliance history with selectable timeframes
[3] Predictions       : LSTM next-hour forecast vs actual comparison
[4] Smart Suggestions : Personalised energy-saving recommendations
[5] Model Metrics     : Live MAE, RMSE, R² per appliance
[6] Architecture      : System design, API docs, project structure

API ENDPOINTS
──────────────
GET  /api/health        → System status
GET  /api/appliances    → List of appliances
GET  /api/stats         → Usage statistics
GET  /api/history/{app} → Historical time series
POST /api/predict       → Single appliance prediction
GET  /api/predict_all   → All appliances predictions
GET  /api/suggestions   → Smart energy suggestions
GET  /api/metrics       → Model evaluation metrics
''')

print('='*70)
print('✓ MILESTONE 4 COMPLETE - SMART ENERGY MONITOR DEPLOYED!')
print('='*70)

---
## 🎉 Week 7-8 Complete!

### What You Built:
| Component | Status |
|-----------|--------|
| Flask REST API (8 endpoints) | ✅ |
| Interactive HTML Dashboard | ✅ |
| Device-wise energy graphs | ✅ |
| LSTM prediction integration | ✅ |
| Smart Suggestions feature | ✅ |
| System architecture docs | ✅ |
| Final project report | ✅ |

### To Present Your Project:
1. Run Step 6 to start the server
2. Open the dashboard URL in browser
3. Walk through each tab: Overview → Devices → Predictions → Suggestions → Metrics → Architecture
4. Show the API working via the test in Step 7